# 🚀 **PASO 1 — Crear catalogo Silver**

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS sdp_catalog;

# 🚀 **PASO 2 — Crear esquemas**

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS sdp_catalog.bronze;

In [0]:
%sql
CREATE SCHEMA  IF NOT EXISTS sdp_catalog.silver;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS sdp_catalog.gold;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS sdp_catalog.semantic;

# 🚀 **PASO 3 — Crear tablas bronze**

In [0]:
%sql
CREATE TABLE sdp_catalog.bronze.claims_raw
(
    claim_id STRING,
    policy_id STRING,
    customer_id STRING,
    claim_date DATE,
    close_date DATE,
    claim_amount DOUBLE,
    claim_status STRING,
    city STRING,
    claim_type STRING
)
USING DELTA;

In [0]:
%sql
CREATE TABLE sdp_catalog.bronze.customers_raw
(
    customer_id STRING,
    customer_name STRING,
    gender STRING,
    birth_date DATE,
    city STRING
)
USING DELTA;

In [0]:
%sql
CREATE TABLE sdp_catalog.bronze.policies_raw
(
    policy_id STRING,
    policy_type STRING,
    premium_amount DOUBLE,
    start_date DATE,
    end_date DATE
)
USING DELTA;

In [0]:
%sql
SHOW TABLES IN sdp_catalog.bronze;

database,tableName,isTemporary
bronze,claims_raw,false
bronze,customers_raw,false
bronze,policies_raw,false


# 🚀 **PASO 4 — Crear tablas Silver**

In [0]:
%sql
CREATE TABLE sdp_catalog.silver.claims_clean
USING DELTA
AS

SELECT DISTINCT
    claim_id,
    policy_id,
    customer_id,
    claim_date,
    close_date,
    claim_amount,
    city,
    claim_type

FROM sdp_catalog.bronze.claims_raw;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE TABLE sdp_catalog.silver.customers_clean
USING DELTA
AS

SELECT DISTINCT
    customer_id,
    customer_name,
    gender,
    birth_date,
    city

FROM sdp_catalog.bronze.customers_raw;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE TABLE sdp_catalog.silver.policies_clean
USING DELTA
AS

SELECT DISTINCT
    policy_id,
    policy_type,
    premium_amount,
    start_date,
    end_date

FROM sdp_catalog.bronze.policies_raw;

num_affected_rows,num_inserted_rows


# 🚀 PASO 5 — Crear Gold Fact Table

In [0]:
%sql
drop TABLE sdp_catalog.gold.fact_claims;
CREATE TABLE sdp_catalog.gold.fact_claims
USING DELTA
AS

SELECT
    c.claim_id,
    c.claim_amount,
    cu.customer_name,
    cu.city,
    p.policy_type

FROM sdp_catalog.silver.claims_clean c

LEFT JOIN sdp_catalog.silver.customers_clean cu
ON c.customer_id = cu.customer_id

LEFT JOIN sdp_catalog.silver.policies_clean p
ON c.policy_id = p.policy_id;

num_affected_rows,num_inserted_rows


# 🚀 PASO 6 — Crear Semantic Layer

**Qué es la capa semántica ?**

Una capa semántica es una interfaz apta para empresas que acorta la brecha entre los modelos de datos complejos y los usuarios empresariales. Actúa como una capa de abstracción que traduce estructuras de datos técnicas en términos y conceptos de negocios familiares, lo que les permite a los analistas de datos y usuarios empresariales obtener, analizar y deducir insights a partir de datos sin requerir una profunda experiencia técnica.

**Definición y propósito**
La capa semántica actúa como una capa de traducción intermediaria en la pila de datos moderna, que transforma los datos sin procesar en información significativa para las empresas. Genera una visión empresarial unificada de los datos de toda la organización, independientemente de dónde se encuentren o cómo estén estructurados técnicamente. Esta abstracción les permite a los analistas de datos enfocarse en generar insights en lugar de lidiar con lenguajes de consulta complejos o comprender esquemas de datos intrincados.

**El rol en la arquitectura de datos**
Dentro de la arquitectura de datos empresarial, la capa semántica se encuentra entre los sistemas de gestión de datos (como almacenes de datos, lagos de datos y data marts) y las herramientas de inteligencia empresarial. Cumple múltiples funciones cruciales en el ecosistema de datos. En primer lugar, estandariza las definiciones y métricas de negocios en toda la organización, lo que asegura la consistencia en la generación de informes y el análisis. Además, gestiona el acceso a los datos y su seguridad, lo que proporciona un entorno seguro para el consumo de datos. La capa también ofrece una interfaz uniforme para herramientas y aplicaciones de análisis, al tiempo que permite una buena gobernanza de datos y mantiene un linaje de datos claro.

# Beneficios empresariales y casos de uso
**Mejora de la calidad y coherencia de los datos**

La capa semántica mejora significativamente la calidad de los datos a través de varios mecanismos críticos. **Establece una única fuente de información para las definiciones de negocios en toda la organización**, lo que asegura que todos los departamentos trabajen desde una misma base de conocimiento. Mediante cálculos y métricas estandarizados, elimina las incoherencias que suelen surgir cuando diferentes equipos interpretan los datos de forma independiente. Esta estandarización se extiende a las políticas de control de datos y crea un marco unificado para la gestión y el uso de los datos.

**Casos de uso en infraestructura de datos moderna**
La capa semántica admite una amplia gama de aplicaciones en entornos de datos modernos. En la elaboración de informes y análisis empresariales, permite una generación de informes coherente en todos los departamentos, manteniendo al mismo tiempo la gobernanza de datos. El análisis multifuncional se vuelve más eficiente cuando los equipos trabajan con las mismas definiciones semánticas. La capa también permite paneles operativos en tiempo real, lo que proporciona información actual sin necesidad de conocimientos técnicos para consultar fuentes de datos en tiempo real. Para proyectos avanzados de análisis y aprendizaje automático, garantiza una ingeniería de características y preparación de datos coherente, lo que acelera el ciclo de desarrollo.

# **Implementación de una capa semántica**
**Pasos para crear e implementar**

Una implementación exitosa de una capa semántica sigue un enfoque estructurado:

- Evaluación de los requisitos y del panorama de datos de la empresa
- Diseño del modelo semántico y definiciones de la empresa
- Configuración de políticas de seguridad y gobernanza
- Integración con herramientas de BI y fuentes de datos
- Pruebas y validación
- Capacitación de usuarios y adopción

In [0]:
%sql
CREATE TABLE IF NOT EXISTS sdp_catalog.semantic.kpi_claims
USING DELTA
AS

SELECT
    city,

    SUM(claim_amount) as total_claims,

    AVG(claim_amount) as avg_claim

FROM sdp_catalog.gold.fact_claims

GROUP BY city;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE TABLE IF NOT EXISTS sdp_catalog.semantic.fraud_claims
USING DELTA
AS
select *

from  sdp_catalog.gold.fact_claims

where claim_amount >
(
    select avg(claim_amount) * 3
    from  sdp_catalog.gold.fact_claims
)

num_affected_rows,num_inserted_rows
